In [ ]:
from core import LADTransferTreeBoost, LSTransferTreeBoost, MTransferTreeBoost
import pandas as pd
from sklearn.datasets import make_friedman1, make_friedman2, make_friedman3
import matplotlib.pyplot as plt
import xgboost as xgb #baseline
from utils import *

In [ ]:
def friedman1(n_samples, noise = 0.0):
    x0 = np.random.uniform(0,1,size=n_samples)
    x1 = np.random.uniform(0,1,size=n_samples)
    x2 = np.random.uniform(0,1,size=n_samples)
    x3 = np.random.uniform(0,1,size=n_samples)
    x4 = np.random.uniform(0,1,size=n_samples)
    X = np.column_stack((x0, x1, x2, x3, x4)) 
    y = 10*np.sin(x0*x1) + 20*(x2 - 0.5)**2 + 10*x3 + 5*x4 + noise*np.random.normal(0,1,size = n_samples) 
    return X, y

def friedman1_altered(n_samples, noise = 0.0):
    x0 = np.random.uniform(0,1,size=n_samples) 
    x1 = np.random.uniform(0,1,size=n_samples)
    x2 = np.random.uniform(0,1,size=n_samples) + 0.2
    x3 = np.random.uniform(0,1,size=n_samples) 
    x4 = np.random.uniform(0,1,size=n_samples)
    X = np.column_stack((x0, x1, x2, x3, x4)) 
    y = 10*np.sin(x0*x1) + 20*(x2 - 0.5)**2 + 10*x3 + 5*x4 + noise*np.random.normal(0,1,size = n_samples)
    return X, y

X, y = friedman1(1000, noise = 0.0)
X_altered, y_altered= friedman1_altered(1000, noise=0.0)

plt.plot(X[:,2], y, '.')
plt.plot(X_altered[:,2], y_altered, '.')
plt.show()


In [ ]:
X_target_test, y_target_test = friedman1(n_samples=10000, noise=0.1) 
X_target_train, y_target_train = friedman1(n_samples=300, noise=0.1)
X_source_train, y_source_train = friedman1_altered(n_samples=10000, noise=0.1) 
X_source_train.shape

In [ ]:
#MTreeBoost is under development
fiter = LSTransferTreeBoost(epochs=200, v=0.2, decay_factor=0.96)
y = np.concatenate((y_source_train, y_target_train))
fiter.fit(X_target_train, y_target_train, X_source_train, y_source_train, show_curves=True)
rmse = fiter.evaluate(X_target_test, y_target_test, metric = 'rmse')
print(rmse)

In [ ]:
params = {
    'objective': 'reg:squarederror',  # Regression with squared error
    'max_depth': 2,                   # Maximum depth of a tree
    'eta': 0.1,                       # Learning rate
    'eval_metric': 'rmse',           # RMSE as evaluation metric
}

def train_xgboost(data_train, labels_train, boosting_rounds):

    dtrain = xgb.DMatrix(data_train, label=labels_train)

    # Train with evaluation set and early stopping
    evallist = [(dtrain, 'train')]

    bst = xgb.train(params, dtrain, num_boost_round=boosting_rounds, evals=evallist)

    return bst

def test_xgboost(data_test, bst):
    
    dtest = xgb.DMatrix(data_test)
    preds = bst.predict(dtest)
    return preds

X = np.concatenate((X_source_train, X_target_train))
bst = train_xgboost(X_target_train, y_target_train, boosting_rounds=200)
preds = test_xgboost(X_target_test, bst)
rmse = compute_rmse(preds, y_target_test)
print(rmse)

In [ ]:
ghgh